In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch

# Competition data
COMP_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

# Saved notebook outputs
NOTEBOOK_ROOT = Path("/kaggle/input/notebooks")

# Find SSL encoder only inside notebook inputs
encoder_files = list(
    NOTEBOOK_ROOT.rglob("best_encoder.pt")
)

assert len(encoder_files) == 1, (
    f"Expected 1 best_encoder.pt, found {len(encoder_files)}"
)

BEST_ENCODER_PATH = encoder_files[0]

# Output folder for this supervised notebook
WORK_DIR = Path("/kaggle/working/supervised")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Competition data:", COMP_ROOT.exists())
print("Best encoder:", BEST_ENCODER_PATH)
print("Best encoder exists:", BEST_ENCODER_PATH.exists())
print("Working directory:", WORK_DIR)

print()
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ============================================================
# Load supervised data
# ============================================================

# Main study-level labels
train_df = pd.read_csv(COMP_ROOT / "train.csv")

# MRI series information
series_df = pd.read_csv(COMP_ROOT / "train_series.csv")

# 12 prediction targets
LABEL_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# Keep only the 58 fully labelled studies
labelled_df = (
    train_df[
        train_df[LABEL_COLS].notna().all(axis=1)
    ]
    .copy()
    .reset_index(drop=True)
)

# MRI series belonging to those 58 studies
labelled_series_df = (
    series_df[
        series_df["StudyInstanceUID"].isin(
            labelled_df["StudyInstanceUID"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("train_df:", train_df.shape)
print("series_df:", series_df.shape)
print("Labelled studies:", len(labelled_df))
print("Labelled MRI series:", len(labelled_series_df))

In [ ]:
# ============================================================
#  Build labelled 2.5D triplets
# ============================================================

from tqdm.auto import tqdm

TRAIN_SERIES_ROOT = COMP_ROOT / "train_series"

def physical_slice_position(ds):
    try:
        orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
        position = np.asarray(ds.ImagePositionPatient, dtype=float)

        row = orientation[:3]
        col = orientation[3:]
        normal = np.cross(row, col)

        return float(np.dot(position, normal))

    except Exception:
        try:
            return float(ds.SliceLocation)
        except Exception:
            return float(ds.InstanceNumber)


# Safe label lookup
label_lookup = (
    labelled_df
    .set_index("StudyInstanceUID")[LABEL_COLS]
    .to_dict(orient="index")
)

triplet_rows = []

series_info = labelled_series_df[
    ["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane"]
]

for study_uid, series_uid, plane in tqdm(
    series_info.itertuples(index=False, name=None),
    total=len(series_info),
    desc="Building labelled triplets"
):

    series_dir = (
        TRAIN_SERIES_ROOT
        / str(study_uid)
        / str(series_uid)
    )

    slices = []

    for file in series_dir.glob("*.dcm"):
        try:
            ds = pydicom.dcmread(
                file,
                stop_before_pixels=True,
                force=True
            )

            slices.append(
                (
                    physical_slice_position(ds),
                    str(file)
                )
            )

        except Exception:
            continue

    slices.sort(key=lambda x: x[0])

    for i in range(1, len(slices) - 1):

        record = {
            "StudyInstanceUID": study_uid,
            "SeriesInstanceUID": series_uid,
            "Anatomical_Plane": plane,
            "PreviousPath": slices[i - 1][1],
            "CentrePath": slices[i][1],
            "NextPath": slices[i + 1][1],
        }

        record.update(label_lookup[study_uid])

        triplet_rows.append(record)


labelled_triplets_df = pd.DataFrame(triplet_rows)

print("Done.")
print("Labelled studies:", labelled_triplets_df["StudyInstanceUID"].nunique())
print("Series:", labelled_triplets_df["SeriesInstanceUID"].nunique())
print("2.5D triplets:", len(labelled_triplets_df))

In [ ]:
# ============================================================
# STEP 4 — Study-level train/validation split + save
# ============================================================

rng = np.random.default_rng(42)

uids = labelled_df["StudyInstanceUID"].to_numpy()
Y = labelled_df[LABEL_COLS].to_numpy(dtype=int)

N_VAL = 12
full_rate = Y.mean(axis=0)

best_score = np.inf
best_val_idx = None

# Search for a reasonably balanced multi-label split
for _ in range(50000):

    val_idx = rng.choice(len(uids), size=N_VAL, replace=False)

    mask = np.zeros(len(uids), dtype=bool)
    mask[val_idx] = True

    y_train = Y[~mask]
    y_val = Y[mask]

    # Require both classes for every target
    if np.any(y_train.sum(axis=0) == 0):
        continue
    if np.any(y_val.sum(axis=0) == 0):
        continue
    if np.any(y_train.sum(axis=0) == len(y_train)):
        continue
    if np.any(y_val.sum(axis=0) == len(y_val)):
        continue

    val_rate = y_val.mean(axis=0)

    score = np.mean(
        ((val_rate - full_rate) ** 2) /
        (full_rate * (1 - full_rate) + 1e-6)
    )

    if score < best_score:
        best_score = score
        best_val_idx = val_idx.copy()


assert best_val_idx is not None

val_uids = set(uids[best_val_idx])

# Assign study-level split
labelled_df["Split"] = np.where(
    labelled_df["StudyInstanceUID"].isin(val_uids),
    "val",
    "train"
)

# Apply same split to all triplets
labelled_triplets_df["Split"] = np.where(
    labelled_triplets_df["StudyInstanceUID"].isin(val_uids),
    "val",
    "train"
)

# Save prepared files
labelled_df.to_csv(
    WORK_DIR / "labelled_studies_with_split.csv",
    index=False
)

labelled_triplets_df.to_parquet(
    WORK_DIR / "labelled_triplets_with_split.parquet",
    index=False
)

print("Train studies:",
      (labelled_df["Split"] == "train").sum())

print("Validation studies:",
      (labelled_df["Split"] == "val").sum())

print()
print("Triplets:")
print(labelled_triplets_df["Split"].value_counts())

print()
print("Saved to:")
print(WORK_DIR)

In [ ]:
# ============================================================
# STEP 5 — Compute labelled-series intensity percentiles
# ============================================================

from tqdm.auto import tqdm

def load_dicom_pixels(path):
    ds = pydicom.dcmread(path, force=True)

    x = ds.pixel_array.astype(np.float32)

    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))

    x = x * slope + intercept

    return x


def compute_series_percentiles(
    study_uid,
    series_uid,
    max_pixels=2_000_000
):
    series_dir = (
        TRAIN_SERIES_ROOT
        / str(study_uid)
        / str(series_uid)
    )

    files = list(series_dir.glob("*.dcm"))

    if len(files) == 0:
        return np.nan, np.nan

    # Deterministic sampling to keep memory under control
    pixels_per_slice = max(
        1,
        max_pixels // len(files)
    )

    samples = []

    for file in files:
        try:
            x = load_dicom_pixels(file).ravel()

            if len(x) > pixels_per_slice:
                idx = np.linspace(
                    0,
                    len(x) - 1,
                    pixels_per_slice,
                    dtype=np.int64
                )
                x = x[idx]

            samples.append(x)

        except Exception:
            continue

    if len(samples) == 0:
        return np.nan, np.nan

    values = np.concatenate(samples)

    p01, p99 = np.percentile(
        values,
        [1, 99]
    )

    return float(p01), float(p99)


percentile_rows = []

unique_series = (
    labelled_series_df[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "Anatomical_Plane"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

for row in tqdm(
    unique_series.itertuples(index=False),
    total=len(unique_series),
    desc="Computing series percentiles"
):

    p01, p99 = compute_series_percentiles(
        row.StudyInstanceUID,
        row.SeriesInstanceUID
    )

    percentile_rows.append({
        "StudyInstanceUID": row.StudyInstanceUID,
        "SeriesInstanceUID": row.SeriesInstanceUID,
        "Anatomical_Plane": row.Anatomical_Plane,
        "P01": p01,
        "P99": p99
    })


labelled_percentiles_df = pd.DataFrame(percentile_rows)

# Check validity
labelled_percentiles_df["Valid"] = (
    labelled_percentiles_df["P01"].notna()
    & labelled_percentiles_df["P99"].notna()
    & (labelled_percentiles_df["P99"] >
       labelled_percentiles_df["P01"])
)

# Save
percentile_path = (
    WORK_DIR / "labelled_series_percentiles.csv"
)

labelled_percentiles_df.to_csv(
    percentile_path,
    index=False
)

print("Series:", len(labelled_percentiles_df))
print()
print("Valid:")
print(labelled_percentiles_df["Valid"].value_counts())
print()
print("Saved:")
print(percentile_path)

In [ ]:
# ============================================================
# STEP 6 — Build final supervised manifest
# ============================================================

supervised_manifest = labelled_triplets_df.merge(
    labelled_percentiles_df[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "P01",
            "P99",
            "Valid"
        ]
    ],
    on=[
        "StudyInstanceUID",
        "SeriesInstanceUID"
    ],
    how="left",
    validate="many_to_one"
)

# Keep only valid series
supervised_manifest = (
    supervised_manifest[
        supervised_manifest["Valid"] == True
    ]
    .copy()
    .reset_index(drop=True)
)

# Save
manifest_path = (
    WORK_DIR / "supervised_manifest.parquet"
)

supervised_manifest.to_parquet(
    manifest_path,
    index=False
)

print("Studies:",
      supervised_manifest["StudyInstanceUID"].nunique())

print("Series:",
      supervised_manifest["SeriesInstanceUID"].nunique())

print("Triplets:",
      len(supervised_manifest))

print()
print("Split:")
print(supervised_manifest["Split"].value_counts())

print()
print("Missing percentiles:",
      supervised_manifest[["P01", "P99"]]
      .isna()
      .sum()
      .sum())

print()
print("Saved:")
print(manifest_path)

In [ ]:
# ============================================================
# STEP 7 — Supervised MRI preprocessing + Dataset
# ============================================================

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

TARGET_SIZE = 224
TARGET_SPACING = 0.5


def read_dicom_for_model(path):
    ds = pydicom.dcmread(
        path,
        force=True
    )

    image = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(ds, "RescaleSlope", 1.0)
    )

    intercept = float(
        getattr(ds, "RescaleIntercept", 0.0)
    )

    image = image * slope + intercept

    # PixelSpacing = [row spacing, column spacing]
    try:
        spacing = ds.PixelSpacing

        row_spacing = float(spacing[0])
        col_spacing = float(spacing[1])

    except Exception:
        # Neutral fallback if spacing is unavailable
        row_spacing = TARGET_SPACING
        col_spacing = TARGET_SPACING

    return (
        image,
        row_spacing,
        col_spacing
    )


def normalize_image(
    image,
    p01,
    p99
):
    image = np.clip(
        image,
        p01,
        p99
    )

    image = (
        (image - p01)
        / (p99 - p01 + 1e-6)
    )

    return image.astype(np.float32)


def resize_to_spacing(
    image,
    row_spacing,
    col_spacing
):
    h, w = image.shape

    new_h = max(
        1,
        int(round(
            h * row_spacing
            / TARGET_SPACING
        ))
    )

    new_w = max(
        1,
        int(round(
            w * col_spacing
            / TARGET_SPACING
        ))
    )

    x = torch.from_numpy(
        image
    ).float()[None, None]

    x = F.interpolate(
        x,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    return x[0, 0]


def centre_crop_or_pad(
    image,
    size=224
):
    h, w = image.shape

    # Pad first if needed
    pad_h = max(
        0,
        size - h
    )

    pad_w = max(
        0,
        size - w
    )

    if pad_h > 0 or pad_w > 0:

        top = pad_h // 2
        bottom = pad_h - top

        left = pad_w // 2
        right = pad_w - left

        image = F.pad(
            image,
            (
                left,
                right,
                top,
                bottom
            )
        )

    # Centre crop
    h, w = image.shape

    y0 = (h - size) // 2
    x0 = (w - size) // 2

    image = image[
        y0:y0 + size,
        x0:x0 + size
    ]

    return image


def preprocess_slice(
    path,
    p01,
    p99
):
    image, row_spacing, col_spacing = (
        read_dicom_for_model(path)
    )

    image = normalize_image(
        image,
        p01,
        p99
    )

    image = resize_to_spacing(
        image,
        row_spacing,
        col_spacing
    )

    image = centre_crop_or_pad(
        image,
        TARGET_SIZE
    )

    return image


def supervised_augmentation(x):
    """
    Mild MRI-safe augmentation.
    x shape: [3, 224, 224]
    """

    # Left/right flip
    if torch.rand(1).item() < 0.5:
        x = torch.flip(
            x,
            dims=[2]
        )

    # Small contrast change
    contrast = (
        0.9
        + torch.rand(1).item() * 0.2
    )

    mean = x.mean()

    x = (
        (x - mean) * contrast
        + mean
    )

    # Small brightness change
    brightness = (
        torch.rand(1).item()
        * 0.08
        - 0.04
    )

    x = x + brightness

    # Light Gaussian noise
    if torch.rand(1).item() < 0.3:

        noise_sigma = (
            torch.rand(1).item()
            * 0.015
        )

        x = (
            x
            + torch.randn_like(x)
            * noise_sigma
        )

    return x.clamp(0.0, 1.0)


class SupervisedKneeDataset(Dataset):

    def __init__(
        self,
        dataframe,
        label_cols,
        augment=False
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.label_cols = label_cols
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        p01 = float(row["P01"])
        p99 = float(row["P99"])

        previous = preprocess_slice(
            row["PreviousPath"],
            p01,
            p99
        )

        centre = preprocess_slice(
            row["CentrePath"],
            p01,
            p99
        )

        next_slice = preprocess_slice(
            row["NextPath"],
            p01,
            p99
        )

        # 2.5D:
        # previous / centre / next = 3 channels
        x = torch.stack(
            [
                previous,
                centre,
                next_slice
            ],
            dim=0
        )

        if self.augment:
            x = supervised_augmentation(x)

        y = torch.tensor(
            row[self.label_cols]
            .to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

        return {
            "image": x,
            "labels": y,
            "study_uid": row[
                "StudyInstanceUID"
            ],
            "series_uid": row[
                "SeriesInstanceUID"
            ],
            "plane": row[
                "Anatomical_Plane"
            ]
        }

In [ ]:
# ============================================================
# STEP 8 — Create supervised datasets
# ============================================================

train_manifest = (
    supervised_manifest[
        supervised_manifest["Split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

val_manifest = (
    supervised_manifest[
        supervised_manifest["Split"] == "val"
    ]
    .copy()
    .reset_index(drop=True)
)


train_dataset = SupervisedKneeDataset(
    train_manifest,
    LABEL_COLS,
    augment=True
)

val_dataset = SupervisedKneeDataset(
    val_manifest,
    LABEL_COLS,
    augment=False
)


print("Train triplets:",
      len(train_dataset))

print("Validation triplets:",
      len(val_dataset))

In [ ]:
# ============================================================
# STEP 9 — Create Study-Balanced DataLoaders
# ============================================================

from torch.utils.data import DataLoader, WeightedRandomSampler

# Number of triplets contributed by each training study
study_counts = (
    train_manifest["StudyInstanceUID"]
    .value_counts()
)

# Give each triplet inverse weight based on its study size
sample_weights = (
    train_manifest["StudyInstanceUID"]
    .map(lambda uid: 1.0 / study_counts[uid])
    .to_numpy(dtype=np.float64)
)

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_manifest),
    replacement=True,
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
# ============================================================
# STEP 10 — Calculate Class Weights
# ============================================================

train_studies_df = (
    labelled_df[
        labelled_df["Split"] == "train"
    ]
    .copy()
)

positive_counts = train_studies_df[LABEL_COLS].sum()
negative_counts = len(train_studies_df) - positive_counts

pos_weight = negative_counts / positive_counts

POS_WEIGHT = torch.tensor(
    pos_weight.to_numpy(dtype=np.float32),
    dtype=torch.float32
)

class_weight_df = pd.DataFrame({
    "Target": LABEL_COLS,
    "Positive": positive_counts.astype(int).values,
    "Negative": negative_counts.astype(int).values,
    "PosWeight": pos_weight.round(3).values
})

print(class_weight_df)

print()
print("POS_WEIGHT shape:", POS_WEIGHT.shape)

class_weight_df.to_csv(
    WORK_DIR / "class_weights.csv",
    index=False
)

In [ ]:
# ============================================================
# STEP 11 — Read Saved SSL Encoder Structure
# ============================================================

checkpoint = torch.load(
    BEST_ENCODER_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    print(list(checkpoint.keys()))

    print("\nFirst parameter names and shapes:")
    for i, (name, value) in enumerate(checkpoint.items()):
        if torch.is_tensor(value):
            print(name, tuple(value.shape))
        elif isinstance(value, dict):
            print(f"{name}: dict with {len(value)} entries")
            for j, (k, v) in enumerate(value.items()):
                if torch.is_tensor(v):
                    print("   ", k, tuple(v.shape))
                if j >= 14:
                    break

        if i >= 14:
            break

In [ ]:
# ============================================================
# STEP 12 — Load SSL encoder + build supervised classifier
# ============================================================

import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Rebuild the same ResNet-18 encoder used during SSL
encoder = resnet18(weights=None)

# SSL encoder had no classification head
encoder.fc = nn.Identity()

# Load learned SSL weights
ssl_state = torch.load(
    BEST_ENCODER_PATH,
    map_location="cpu",
    weights_only=True
)

encoder.load_state_dict(
    ssl_state,
    strict=True
)

print("✓ SSL encoder loaded")


class KneeClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        num_labels=12,
        dropout=0.3
    ):
        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, num_labels)
        )

    def forward(self, x):

        features = self.encoder(x)

        logits = self.classifier(features)

        return logits


model = KneeClassifier(
    encoder=encoder,
    num_labels=len(LABEL_COLS)
)

model = model.to(device)

print("Device:", device)
print("Output labels:", len(LABEL_COLS))

In [ ]:
# ============================================================
# STEP 13 — Loss + optimizer
# ============================================================

POS_WEIGHT = POS_WEIGHT.to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=POS_WEIGHT
)

# Small learning rate for pretrained encoder
# Larger learning rate for new classifier head
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.encoder.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

EPOCHS = 10

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Loss: BCEWithLogitsLoss")
print("Epochs:", EPOCHS)
print("Encoder LR:", optimizer.param_groups[0]["lr"])
print("Classifier LR:", optimizer.param_groups[1]["lr"])

In [ ]:
# ============================================================
# STEP 14 — Supervised fine-tuning
# ============================================================

from tqdm.auto import tqdm
import pandas as pd

BEST_MODEL_PATH = WORK_DIR / "best_supervised_model.pt"
HISTORY_PATH = WORK_DIR / "supervised_training_history.csv"

USE_AMP = device.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

best_val_loss = float("inf")
patience = 3
epochs_without_improvement = 0

history = []


# ------------------------------------------------------------
# Study-level validation
# ------------------------------------------------------------

def validate_study_level(model, loader):

    model.eval()

    all_logits = []
    all_labels = []
    all_studies = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc="Validation",
            leave=False
        ):

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type=device.type,
                enabled=USE_AMP
            ):
                logits = model(images)

            all_logits.append(
                logits.detach().cpu()
            )

            all_labels.append(
                labels.detach().cpu()
            )

            all_studies.extend(
                batch["study_uid"]
            )

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    # --------------------------------------------------------
    # Aggregate triplet predictions to study level
    # --------------------------------------------------------

    study_logits = []
    study_labels = []

    unique_studies = list(
        dict.fromkeys(all_studies)
    )

    for study_uid in unique_studies:

        indices = [
            i
            for i, uid in enumerate(all_studies)
            if uid == study_uid
        ]

        indices = torch.tensor(
            indices,
            dtype=torch.long
        )

        # Mean prediction across all triplets from the study
        mean_logits = all_logits[
            indices
        ].mean(dim=0)

        label = all_labels[
            indices[0]
        ]

        study_logits.append(mean_logits)
        study_labels.append(label)

    study_logits = torch.stack(
        study_logits
    ).to(device)

    study_labels = torch.stack(
        study_labels
    ).to(device)

    val_loss = criterion(
        study_logits,
        study_labels
    )

    return float(val_loss.item())


# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    print()
    print("=" * 65)
    print(f"EPOCH {epoch}/{EPOCHS}")
    print("=" * 65)

    model.train()

    running_loss = 0.0
    samples_seen = 0

    train_bar = tqdm(
        train_loader,
        desc="Training"
    )

    for batch in train_bar:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP
        ):

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )

        samples_seen += batch_size

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss
        / samples_seen
    )

    # --------------------------------------------------------
    # Study-level validation
    # --------------------------------------------------------

    val_loss = validate_study_level(
        model,
        val_loader
    )

    scheduler.step()

    encoder_lr = (
        optimizer.param_groups[0]["lr"]
    )

    classifier_lr = (
        optimizer.param_groups[1]["lr"]
    )

    print()
    print(f"Train loss:       {train_loss:.4f}")
    print(f"Study val loss:   {val_loss:.4f}")
    print(f"Encoder LR:       {encoder_lr:.8f}")
    print(f"Classifier LR:    {classifier_lr:.8f}")

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "encoder_lr": encoder_lr,
        "classifier_lr": classifier_lr
    })

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_loss":
                    best_val_loss,

                "label_cols":
                    LABEL_COLS
            },
            BEST_MODEL_PATH
        )

        print("✓ New best supervised model saved")

    else:

        epochs_without_improvement += 1

        print(
            "No improvement:",
            f"{epochs_without_improvement}/{patience}"
        )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= patience:

        print()
        print("Early stopping triggered.")
        break


print()
print("=" * 65)
print("SUPERVISED FINE-TUNING COMPLETE")
print("=" * 65)

print(
    "Best study-level validation loss:",
    round(best_val_loss, 4)
)

print("Best model:")
print(BEST_MODEL_PATH)

In [ ]:
# ============================================================
# STEP 15 — Load best supervised model
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("✓ Best supervised model loaded")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Model path:", BEST_MODEL_PATH)

In [ ]:
# ============================================================
# STEP 16 — Final study-level validation evaluation
# ============================================================

from sklearn.metrics import roc_auc_score

model.eval()

study_logit_sum = {}
study_count = {}
study_labels = {}

with torch.no_grad():

    for batch in tqdm(
        val_loader,
        desc="Final validation"
    ):

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"]

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)

        logits = logits.cpu()

        for uid, logit, label in zip(
            batch["study_uid"],
            logits,
            labels
        ):

            if uid not in study_logit_sum:

                study_logit_sum[uid] = (
                    logit.clone()
                )

                study_count[uid] = 1

                study_labels[uid] = (
                    label.clone()
                )

            else:

                study_logit_sum[uid] += logit
                study_count[uid] += 1


# ------------------------------------------------------------
# Aggregate triplets -> study
# ------------------------------------------------------------

val_rows = []

all_study_logits = []
all_study_labels = []

for uid in study_logit_sum:

    mean_logit = (
        study_logit_sum[uid]
        / study_count[uid]
    )

    probs = torch.sigmoid(
        mean_logit
    )

    labels = study_labels[uid]

    all_study_logits.append(
        mean_logit
    )

    all_study_labels.append(
        labels
    )

    row = {
        "StudyInstanceUID": uid
    }

    for i, target in enumerate(LABEL_COLS):

        row[f"{target}_true"] = (
            float(labels[i])
        )

        row[f"{target}_prob"] = (
            float(probs[i])
        )

    val_rows.append(row)


val_predictions_df = pd.DataFrame(
    val_rows
)

all_study_logits = torch.stack(
    all_study_logits
).to(device)

all_study_labels = torch.stack(
    all_study_labels
).to(device)

final_val_loss = criterion(
    all_study_logits,
    all_study_labels
).item()


# ------------------------------------------------------------
# Per-target AUROC
# ------------------------------------------------------------

metric_rows = []

for target in LABEL_COLS:

    y_true = (
        val_predictions_df[
            f"{target}_true"
        ].values
    )

    y_prob = (
        val_predictions_df[
            f"{target}_prob"
        ].values
    )

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    metric_rows.append({
        "Target": target,
        "AUROC": auc
    })


validation_metrics_df = pd.DataFrame(
    metric_rows
)

mean_auc = (
    validation_metrics_df["AUROC"]
    .mean()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

val_predictions_df.to_csv(
    WORK_DIR
    / "validation_predictions.csv",
    index=False
)

validation_metrics_df.to_csv(
    WORK_DIR
    / "validation_metrics.csv",
    index=False
)


print("Validation studies:", len(val_predictions_df))
print("Study-level validation loss:", round(final_val_loss, 4))
print("Mean AUROC:", round(mean_auc, 4))

print()
print(validation_metrics_df)

In [ ]:
# ============================================================
# STEP 17 — Test inference + submission
# ============================================================

from torch.utils.data import Dataset, DataLoader

TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
TEST_SERIES_ROOT = COMP_ROOT / "test_series"
SAMPLE_SUBMISSION_PATH = (
    COMP_ROOT / "sample_submission.csv"
)

assert TEST_SERIES_CSV.exists()
assert TEST_SERIES_ROOT.exists()
assert SAMPLE_SUBMISSION_PATH.exists()


# ------------------------------------------------------------
# Load competition test structure
# ------------------------------------------------------------

test_series_df = pd.read_csv(
    TEST_SERIES_CSV
)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH
)

assert "StudyInstanceUID" in sample_submission.columns

for target in LABEL_COLS:
    assert target in sample_submission.columns


submission_uids = set(
    sample_submission["StudyInstanceUID"]
)


test_series_df = (
    test_series_df[
        test_series_df[
            "StudyInstanceUID"
        ].isin(submission_uids)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Test studies:",
    len(submission_uids)
)

print(
    "Test MRI series:",
    len(test_series_df)
)


# ------------------------------------------------------------
# Build clean test 2.5D manifest
# ------------------------------------------------------------

test_triplet_rows = []
test_percentile_rows = []

series_info = test_series_df[
    [
        "StudyInstanceUID",
        "SeriesInstanceUID",
        "Anatomical_Plane"
    ]
].drop_duplicates()


for study_uid, series_uid, plane in tqdm(
    series_info.itertuples(
        index=False,
        name=None
    ),
    total=len(series_info),
    desc="Preparing test MRI series"
):

    series_dir = (
        TEST_SERIES_ROOT
        / str(study_uid)
        / str(series_uid)
    )

    files = list(
        series_dir.glob("*.dcm")
    )

    if len(files) < 3:
        continue

    slices = []
    pixel_samples = []

    pixels_per_slice = max(
        1,
        2_000_000 // len(files)
    )

    for file in files:

        try:

            ds = pydicom.dcmread(
                file,
                force=True
            )

            position = (
                physical_slice_position(ds)
            )

            image = (
                ds.pixel_array
                .astype(np.float32)
            )

            slope = float(
                getattr(
                    ds,
                    "RescaleSlope",
                    1.0
                )
            )

            intercept = float(
                getattr(
                    ds,
                    "RescaleIntercept",
                    0.0
                )
            )

            image = (
                image * slope
                + intercept
            )

            flat = image.ravel()

            if len(flat) > pixels_per_slice:

                idx = np.linspace(
                    0,
                    len(flat) - 1,
                    pixels_per_slice,
                    dtype=np.int64
                )

                flat = flat[idx]

            pixel_samples.append(
                flat
            )

            slices.append(
                (
                    position,
                    str(file)
                )
            )

        except Exception:
            continue


    if (
        len(slices) < 3
        or len(pixel_samples) == 0
    ):
        continue


    # Physical slice order
    slices.sort(
        key=lambda x: x[0]
    )


    values = np.concatenate(
        pixel_samples
    )

    p01, p99 = np.percentile(
        values,
        [1, 99]
    )

    p01 = float(p01)
    p99 = float(p99)

    if (
        not np.isfinite(p01)
        or not np.isfinite(p99)
        or p99 <= p01
    ):
        continue


    test_percentile_rows.append({
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "Anatomical_Plane": plane,
        "P01": p01,
        "P99": p99,
        "ValidSlices": len(slices)
    })


    # Previous / centre / next
    for i in range(
        1,
        len(slices) - 1
    ):

        test_triplet_rows.append({

            "StudyInstanceUID":
                study_uid,

            "SeriesInstanceUID":
                series_uid,

            "Anatomical_Plane":
                plane,

            "PreviousPath":
                slices[i - 1][1],

            "CentrePath":
                slices[i][1],

            "NextPath":
                slices[i + 1][1],

            "P01":
                p01,

            "P99":
                p99
        })


test_manifest = pd.DataFrame(
    test_triplet_rows
)

test_percentiles_df = pd.DataFrame(
    test_percentile_rows
)


print()
print(
    "Prepared test studies:",
    test_manifest[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Prepared test series:",
    test_manifest[
        "SeriesInstanceUID"
    ].nunique()
)

print(
    "Test triplets:",
    len(test_manifest)
)


# Save manifests
test_manifest.to_parquet(
    WORK_DIR
    / "test_manifest.parquet",
    index=False
)

test_percentiles_df.to_csv(
    WORK_DIR
    / "test_series_percentiles.csv",
    index=False
)


# ------------------------------------------------------------
# Test Dataset
# ------------------------------------------------------------

class TestKneeDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )


    def __len__(self):
        return len(self.df)


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        p01 = float(
            row["P01"]
        )

        p99 = float(
            row["P99"]
        )


        previous = preprocess_slice(
            row["PreviousPath"],
            p01,
            p99
        )

        centre = preprocess_slice(
            row["CentrePath"],
            p01,
            p99
        )

        next_slice = preprocess_slice(
            row["NextPath"],
            p01,
            p99
        )


        image = torch.stack(
            [
                previous,
                centre,
                next_slice
            ],
            dim=0
        )


        return {
            "image": image,

            "study_uid":
                row[
                    "StudyInstanceUID"
                ]
        }


test_dataset = TestKneeDataset(
    test_manifest
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        device.type == "cuda"
    )
)


# ------------------------------------------------------------
# GPU inference
# ------------------------------------------------------------

model.eval()

study_logit_sum = {}
study_count = {}


with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Test inference"
    ):

        images = (
            batch["image"]
            .to(
                device,
                non_blocking=True
            )
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(
                device.type == "cuda"
            )
        ):

            logits = model(
                images
            )

        logits = logits.cpu()


        for uid, logit in zip(
            batch["study_uid"],
            logits
        ):

            if uid not in study_logit_sum:

                study_logit_sum[uid] = (
                    logit.clone()
                )

                study_count[uid] = 1

            else:

                study_logit_sum[uid] += (
                    logit
                )

                study_count[uid] += 1


# ------------------------------------------------------------
# Convert study logits -> probabilities
# ------------------------------------------------------------

prediction_rows = []


for uid in study_logit_sum:

    mean_logit = (
        study_logit_sum[uid]
        / study_count[uid]
    )

    probabilities = torch.sigmoid(
        mean_logit
    ).numpy()


    row = {
        "StudyInstanceUID": uid
    }


    for i, target in enumerate(
        LABEL_COLS
    ):

        row[target] = float(
            probabilities[i]
        )


    prediction_rows.append(
        row
    )


test_predictions_df = pd.DataFrame(
    prediction_rows
)


# ------------------------------------------------------------
# Match exact Kaggle submission ordering
# ------------------------------------------------------------

submission = (
    sample_submission[
        ["StudyInstanceUID"]
    ]
    .merge(
        test_predictions_df,
        on="StudyInstanceUID",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Final safety checks
# ------------------------------------------------------------

assert len(submission) == len(
    sample_submission
)

assert submission[
    LABEL_COLS
].isna().sum().sum() == 0

assert np.isfinite(
    submission[
        LABEL_COLS
    ].to_numpy()
).all()

assert (
    submission[
        LABEL_COLS
    ].to_numpy() >= 0
).all()

assert (
    submission[
        LABEL_COLS
    ].to_numpy() <= 1
).all()


# Restore exact sample submission column order
submission = submission[
    sample_submission.columns
]


SUBMISSION_PATH = (
    WORK_DIR
    / "submission.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)


print()
print("=" * 65)
print("TEST INFERENCE COMPLETE")
print("=" * 65)

print(
    "Predicted studies:",
    len(submission)
)

print(
    "Missing predictions:",
    submission[
        LABEL_COLS
    ].isna().sum().sum()
)

print()
print("Submission saved:")
print(SUBMISSION_PATH)

print()
print(submission.head())